# 04.02 · Proyecto integrador desde cero

**Pregunta:** ¿Qué clientes/mercados priorizar y cómo influyen cambios de costos? Recorrido: carga → control → uniones → indicadores → escenarios → decisiones.

Los registros son **sintéticos**; un escenario no es un pronóstico de negocio.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
base=Path('../../00 Fundamentos de Datos - Excel y SQL/datos') if Path('../../00 Fundamentos de Datos - Excel y SQL/datos').exists() else Path('00 Fundamentos de Datos - Excel y SQL/datos')
v=pd.read_csv(base/'ventas.csv',parse_dates=['fecha']);c=pd.read_csv(base/'clientes.csv');p=pd.read_csv(base/'productos.csv')
assert v.id.is_unique and c.cliente_id.is_unique and p.producto_id.is_unique
assert v.cliente_id.isin(c.cliente_id).all() and v.producto_id.isin(p.producto_id).all()
df=v.merge(c,on='cliente_id',validate='many_to_one').merge(p,on='producto_id',validate='many_to_one')
df['ingreso']=df.unidades*df.precio_unitario
df['costo']=df.unidades*df.costo_unitario
df['margen']=df.ingreso-df.costo
print('Filas:',len(df),'Nulos:',int(df.isna().sum().sum()))

## Diagnóstico de clientes
Calcula ingreso, margen, cantidad de operaciones, primera/última compra. Esta muestra cubre 12 meses ficticios; recencia depende de la fecha de corte seleccionada.

In [ ]:
corte=df.fecha.max()
clientes=df.groupby(['cliente_id','cliente','pais','segmento']).agg(ingreso=('ingreso','sum'),margen=('margen','sum'),operaciones=('id','count'),primera_compra=('fecha','min'),ultima_compra=('fecha','max')).reset_index()
clientes['dias_desde_ultima']=(corte-clientes.ultima_compra).dt.days
clientes=clientes.sort_values('ingreso',ascending=False)
clientes

## Escenarios de costos
Los costos fijos, la nómina, certificaciones y fletes reales no están incluidos. La tabla solo ilustra sensibilidad sobre el costo variable registrado.

In [ ]:
escenarios=[]
for aumento in [0,0.05,0.10,0.20]:
    ingreso=df.ingreso.sum();costo=df.costo.sum()*(1+aumento)
    escenarios.append({'aumento_costos_pct':100*aumento,'ingreso':ingreso,'costo_escenario':costo,'margen_escenario':ingreso-costo,'margen_pct':100*(ingreso-costo)/ingreso})
pd.DataFrame(escenarios)

## Pregunta difícil
Un cliente grande puede traer ingreso elevado y al mismo tiempo exigir descuentos, lotes especiales o tiempos de entrega riesgosos. ¿Cómo cambiaría tu recomendación al introducir costos de logística y calidad?

In [ ]:
tabla=df.groupby(['pais','producto']).agg(ingresos=('ingreso','sum'),margen=('margen','sum')).reset_index()
print(tabla.sort_values('margen',ascending=False).head(10).to_string(index=False))
tabla.pivot(index='producto',columns='pais',values='margen').fillna(0).plot.bar(title='Margen bruto por producto y mercado');plt.tight_layout();plt.show()

## Entrega y criterios de aceptación
1. Escribe 5 preguntas de negocio antes de analizar.
2. Entrega auditoría de calidad con checks que fallen cuando los datos son incorrectos.
3. Construye modelo equivalente en SQL PostgreSQL y prueba al menos 4 consultas.
4. Haz 2 escenarios de riesgo adicionales explicando hipótesis.
5. Prepara dashboard Streamlit, README de uso y memo ejecutivo con evidencias.
6. Discute si harías una llamada en frío a un cliente priorizado: qué sabes, qué desconoces y qué pregunta abriría la conversación.
7. Separa resultados descriptivos, interpretaciones y recomendaciones. No atribuyas causalidad.

**Bonus:** agrega una tabla de logística ficticia por país y recalcula contribución con fletes.